# 05 — Séries Temporais

Funções: `testa_estacionaridade`, `autocorrelation_lag`, `cross_correlation_lag`, `teste_causalidade`, `g_previsoes_series_temporais`, `originalVStreino`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## 1. `testa_estacionaridade`

Plota a média e o desvio padrão móveis (janela 12) e executa o **Teste Dickey-Fuller Aumentado**: valor-p < 0.05 indica série estacionária.

In [ ]:
from ML.supervisionado.series_temporais.suposicoes_series_temporais import testa_estacionaridade

np.random.seed(0)
datas = pd.date_range('2022-01-01', periods=120, freq='ME')

# Serie NAO estacionaria (tendencia crescente)
serie_trend = pd.Series(
    np.linspace(100, 200, 120) + np.random.normal(0, 5, 120),
    index=datas, name='com_tendencia'
)
print("=== Serie com tendencia (nao estacionaria) ===")
testa_estacionaridade(serie_trend)

# Serie estacionaria (sem tendencia)
serie_flat = pd.Series(
    150 + np.random.normal(0, 5, 120),
    index=datas, name='estacionaria'
)
print("\n=== Serie estacionaria ===")
testa_estacionaridade(serie_flat)


## 2. `autocorrelation_lag`

Calcula a autocorrelação e retorna o **lag de máxima autocorrelação** (excluindo lag=0). Útil para identificar sazonalidade e ordem de modelos AR.

In [ ]:
from ML.supervisionado.series_temporais.suposicoes_series_temporais import autocorrelation_lag

np.random.seed(1)
t = np.arange(150)

# Serie com sazonalidade semanal (lag esperado proximo de 7)
serie_saz = pd.Series(50 + 10 * np.sin(2 * np.pi * t / 7) + np.random.normal(0, 2, 150))

lag_max, _ = autocorrelation_lag(serie_saz, plot=True)
print(f"\nLag de maxima autocorrelacao: {lag_max}")
plt.show()


## 3. `cross_correlation_lag`

Calcula a **correlação cruzada** entre duas séries e retorna o lag em que a correlação é máxima. Útil para medir o atraso de resposta de uma série em relação a outra (ex: marketing -> vendas).

In [ ]:
from ML.supervisionado.series_temporais.suposicoes_series_temporais import cross_correlation_lag

np.random.seed(2)
x = pd.Series(np.random.randn(100))

# y reage a x com lag 5 (com ruido)
y = x.shift(5).fillna(0) + np.random.normal(0, 0.3, 100)

lag_max, _ = cross_correlation_lag(x, y, plot=True)
print(f"\nLag de maxima correlacao cruzada: {lag_max}")
plt.show()


## 4. `teste_causalidade` (Granger)

Testa se a série `X` **Granger-causa** a série `Y` para cada lag de 1 a `max_lag`. Valor-p < 0.05 sugere que X contém informação preditiva sobre Y.

In [ ]:
from ML.supervisionado.series_temporais.suposicoes_series_temporais import teste_causalidade

np.random.seed(3)
n = 120
marketing = np.cumsum(np.random.randn(n))
vendas    = np.cumsum(np.random.randn(n))

# Adiciona causalidade artificial: vendas reage ao marketing com lag 2
vendas += pd.Series(marketing).shift(2).fillna(0).values * 0.6

resultados = teste_causalidade(marketing, vendas, max_lag=5)


## 5. Visualizações de previsões

- `g_previsoes_series_temporais`: plota treino, validação e previsões no mesmo gráfico  
- `originalVStreino`: compara a série original completa com as previsões

In [ ]:
from ML.supervisionado.series_temporais.validacao_series_temporais import (
    g_previsoes_series_temporais,
    originalVStreino,
)

np.random.seed(0)
datas_treino = pd.date_range('2024-01-01', periods=80, freq='D')
datas_valid  = pd.date_range('2024-03-21', periods=20, freq='D')

df_treino = pd.Series(np.cumsum(np.random.randn(80)) + 100, index=datas_treino)
df_valid  = pd.Series(np.cumsum(np.random.randn(20)) + 110, index=datas_valid)
previsoes  = df_valid + np.random.normal(0, 3, 20)

g_previsoes_series_temporais(df_treino, df_valid, previsoes, modelo='SARIMA')
originalVStreino(pd.concat([df_treino, df_valid]), previsoes)
